In [2]:
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objs as go

import numpy as np
from scipy.stats import gaussian_kde
import plotly.graph_objects as go

# ETL

In [131]:
# TODO: user input for group name
GNAME = "Grupo Deus é Amor" # TODO

conn = sqlite3.connect('dataset/railway.sqlite')
cursor = conn.cursor()

# Get all tags for a group
query_tags = "SELECT name FROM tags WHERE group_id = (SELECT id FROM groups WHERE name = ?)"
cursor.execute(query_tags, (GNAME,))
tags = cursor.fetchall()
tags = [tag[0] for tag in tags]

# TODO: drop down menu for tags

# TODO: user input for tag selection(s)
TAG = ['quinta'] # TODO

In [132]:
def get_placeholder(target_list):
    return ','.join(['?'] * len(target_list))

# Get all event ids for the group with the specified tags
placeholder_tags = get_placeholder(TAG)
query_eid = f"\
    SELECT DISTINCT e.id \
    FROM events AS e \
    JOIN event_tags AS et ON et.event_id = e.id \
    JOIN tags ON tags.id = et.tag_id \
    WHERE e.group_id = (SELECT id FROM groups WHERE name = ?)\
        AND tags.name IN ({placeholder_tags})"
    
cursor.execute(query_eid, [GNAME]+TAG)
event_ids = cursor.fetchall()
event_ids = [eid[0] for eid in event_ids]

# Get participant & checkin info for the events
placeholder_eids = get_placeholder(event_ids)
query = f"\
    SELECT e.id AS event_id, e.name AS event_name, e.start_date_time AS event_time, \
        p.id AS participant_id, p.full_name AS participant_name, c.timestamp AS checkin_time, \
        p.birth_date AS participant_birth, p.gender AS participant_gender \
    FROM events AS e \
    JOIN check_ins AS c ON c.event_id = e.id \
    JOIN participants AS p ON p.id = c.participant_id \
    WHERE e.id IN ({placeholder_eids})" 
# cursor.execute(query, event_ids)
# results = cursor.fetchall()
df = pd.read_sql_query(query, conn, params=event_ids)

In [100]:
### === Data Cleaning and Preprocessing ===

# # Check for missing values
# df.isnull().sum()
# df.isna().sum()

# Convert date columns to datetime format
for col in ["event_time", "checkin_time", "participant_birth"]:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Adjust time zone
df['checkin_time'] = df['checkin_time'].dt.tz_localize('UTC').dt.tz_convert('Brazil/East')
    
# Add age column
df["participant_age"] = (pd.to_datetime("today") - df["participant_birth"]).dt.days // 365

# De-identify participants by only keeping first names
df["participant_name"] = df["participant_name"].str.split().str[0]

In [101]:
# # abnormal age value check
# abnormal_age_ids = df[df["participant_age"] <= 7]["participant_id"].drop_duplicates().tolist()

# # Get contact info for abnormal age participants
# placeholders = ",".join("?" * len(abnormal_age_ids))
# query = f"SELECT id, full_name, email, phone, birth_date FROM participants WHERE id IN ({placeholders})"

# contact_info = pd.read_sql(query, conn, params=abnormal_age_ids)
# contact_info.to_csv("dataset/abnormal_age_participants.csv", index=False)

conn.close()

In [19]:
df.dtypes

event_id                                   object
event_name                                 object
event_time                         datetime64[ns]
participant_id                             object
participant_name                           object
checkin_time          datetime64[ns, Brazil/East]
participant_birth                  datetime64[ns]
participant_gender                         object
participant_age                             int64
dtype: object

# EDA

## Group Participants Analysis

In [102]:
# all registered participants in the group
participants = df[["participant_id", "participant_name", "participant_gender", "participant_age"]].drop_duplicates().reset_index(drop=True)

# filtering age out of bound participants
participants = participants[(participants["participant_age"] > 7) & (participants["participant_age"] < 30)].reset_index(drop=True)

# add attendance count
attendance = df.groupby("participant_id").size().reset_index(name="attendance_count")
participants = participants.merge(attendance, on="participant_id", how="left")

# rename columns for better readability
participants.columns = ["id", "name", "gender", "age", "attendance_count"]
participants.head(3)

,id,name,gender,age,attendance_count
0,59fa3bcd-4903-4349-ac44-4af3352efc93,Felipe,MALE,16,2
1,346c4a15-bf18-4917-9877-48df46d84fdc,Felipe,MALE,13,1
2,13762581-ab39-4b01-ab0b-274ac8c6bceb,Antônio,MALE,17,3


In [173]:
# number of participants
num_participants = participants['id'].nunique()
print(f"Total participants: {num_participants}")

Total participants: 680


### Gender & Age Analysis

In [ ]:
# Gender distribution
gender = participants["gender"]
gender_info = pd.DataFrame({'count': gender.value_counts(), 
                            'percentage': (gender.value_counts(normalize=True) * 100).round(2)})
gender_info.reset_index(inplace=True)
gender_pct = {"male": gender_info[gender_info["gender"] == "MALE"]["percentage"].iloc[0],
              "female": gender_info[gender_info["gender"] == "FEMALE"]["percentage"].iloc[0]} # for dashboard
gender_info

,gender,count,percentage
0,FEMALE,366,53.82
1,MALE,314,46.18


In [174]:
# Age distribution (overall)
def get_age_info(age_data, freq=False):
    mean, median, min, max = age_data.mean(), age_data.median(), age_data.min(), age_data.max()
    print(f"Average age: {mean:.1f} \nMedian age: {median} \nAge range: {min} - {max}")
    if freq:
        print("\nMost frequent ages:")
        values = age_data.value_counts().reset_index()
        for row in values.head(3).itertuples():
            print(f"Age {row[1]}: {row[2]} participants")
    return {"mean": mean, "median": median, "min": min, "max": max}

overall_age = participants['age']
age_info = get_age_info(overall_age)

Average age: 14.7 
Median age: 15.0 
Age range: 11 - 25


In [105]:
age_df = overall_age.value_counts().reset_index()
# (age_count/age_count.sum()).cumsum()
age_df['percentage'] = age_df['count']/age_df['count'].sum()
age_df['cumulative_percentage'] = age_df['percentage'].cumsum()

# pareto = go.Figure([
#     go.Bar(x=age_df['age'], y=age_df['count'], name='Age Count'),
#     go.Scatter(x=age_df['age'], y=age_df['cumulative_percentage'], 
#                mode='lines+markers', yaxis='y2', name='Cumulative %')]) 
# pareto = pareto.update_layout(yaxis2=dict(overlaying='y', side='right', tickformat='.0%', range=[0, 1]))
# pareto.add_hline(y=0.8, yref='y2', line_dash="dash")

age_df.head()

,age,count,percentage,cumulative_percentage
0,14,161,0.236765,0.236765
1,15,154,0.226471,0.463235
2,13,127,0.186765,0.650000
3,16,110,0.161765,0.811765
4,12,42,0.061765,0.873529


More than 80% of all participants are of age 13 - 16

In [192]:
bar_age = px.bar(age_df, x='age', y='count', text='count',
       title="Age Distribution of Participants")
# hist_age = px.histogram(participants, x="age", 
#              title="Age Distribution of Participants")
bar_age.update_layout(bargap=0.2, width=600, height=450)
bar_age.update_xaxes(dtick=1)
bar_age.update_traces(marker=dict(color="#dd9add", line=dict(color='black', width=1)))

In [107]:
age_gender = participants.groupby(["gender", "age"], as_index=False).size()
age_gender.sort_values(by=['size'], inplace=True, ascending=False)
age_gender['pct'] = age_gender['size'] / age_gender['size'].sum()
age_gender['cumulative_pct'] = age_gender['pct'].cumsum()

age_gender.head()

,gender,age,size,pct,cumulative_pct
2,FEMALE,13,90,0.132353,0.132353
3,FEMALE,14,87,0.127941,0.260294
12,MALE,15,83,0.122059,0.382353
11,MALE,14,74,0.108824,0.491176
4,FEMALE,15,71,0.104412,0.595588


In [108]:
female_age = participants[participants["gender"] == "FEMALE"]["age"]
male_age = participants[participants["gender"] == "MALE"]["age"]
print("Female Participants Age Info:")
get_age_info(female_age, freq=True)
print("\nMale Participants Age Info:")
get_age_info(male_age, freq=True)


Female Participants Age Info:
Average age: 14.4 
Median age: 14.0 
Age range: 11 - 19

Most frequent ages:
Age 13: 90 participants
Age 14: 87 participants
Age 15: 71 participants

Male Participants Age Info:
Average age: 15.2 
Median age: 15.0 
Age range: 12 - 25

Most frequent ages:
Age 15: 83 participants
Age 14: 74 participants
Age 16: 56 participants


In [199]:
# histogram: age distribution by gender 
hist_age_gender = px.histogram(participants, x='age',
             color='gender',
             barmode='group', # 'group' - side by side, 'stack' - stacked bars
             # histnorm='probability density',
             title='Age Distribution By Gender',
             color_discrete_map={'MALE': '#87ceeb', 'FEMALE': '#ffb6c1'})
hist_age_gender.update_layout(bargap=0.3, #width=600, height=400
                              autosize=True, margin=dict(l=20, r=20, t=40, b=20))
hist_age_gender.update_traces(marker=dict(line=dict(color='black', width=1)))
hist_age_gender.update_xaxes(dtick=1)

# # Add KDE automatically per gender
# for gender, group in participants.groupby('gender'):
#     ages = group['age'].dropna()
#     kde = gaussian_kde(ages)
#     x_range = np.linspace(ages.min(), ages.max(), 100)
#     hist.add_trace(go.Scatter(x=x_range, y=kde(x_range), mode='lines', name=f'{gender} curve'))


In [168]:
# px.violin(participants, x="gender", y="age", 
#           color="gender", box=True,
#           title="Age Distribution by Gender")

### Attendance Analysis

In [110]:
num_events = df["event_id"].nunique()
participants["attendance %"] = participants["attendance_count"] / num_events * 100
participants.head(3)

,id,name,gender,age,attendance_count,attendance %
0,59fa3bcd-4903-4349-ac44-4af3352efc93,Felipe,MALE,16,2,40.0
1,346c4a15-bf18-4917-9877-48df46d84fdc,Felipe,MALE,13,1,20.0
2,13762581-ab39-4b01-ab0b-274ac8c6bceb,Antônio,MALE,17,3,60.0


In [111]:
# Full attendance
full_att_pct = len(participants[participants["attendance_count"] == num_events]) / len(participants)
print(f"Full attendance rate: {full_att_pct:.2%}")

Full attendance rate: 6.76%


In [112]:
# gender vs attendance
att_by_gender = participants.groupby("gender")["attendance %"]
print(att_by_gender.agg(['mean', 'std', 'var']))
# px.box(participants, x="gender", y="attendance_count", color="gender") ## not useful, two boxes are the same
# px.histogram(participants, x="attendance_count", barmode="group", color="gender", opacity=0.75) ## not useful, more female than male, comparison not meaningful

             mean        std         var
gender                                  
FEMALE  45.027322  26.666367  711.095142
MALE    38.853503  24.637128  606.988055


In [113]:
# age vs attendance
att_by_age = participants.groupby("age")["attendance_count"]
# print(att_by_age.mean())
participants[['age', 'attendance_count']].corr()

,age,attendance_count
age,1.000000,-0.037177
attendance_count,-0.037177,1.000000


In [114]:
px.scatter(participants, x='age', y='attendance %',
           color='gender',  # optional, adds gender context
           trendline='ols') # adds a regression trendline

In [ ]:
# att_by_age.mean().sort_values(ascending=False)
# px.bar(att_by_age.mean().reset_index(), x='age', y='attendance_count')

In [115]:
att_age_gender = participants.groupby(['gender','age'])['attendance %']
engagement = att_age_gender.agg(['mean', 'count']).reset_index()
engagement = engagement[engagement['count'] >= 5] # remove groups too small to be statistically meaningful.
engagement = engagement.sort_values(by=['mean'], ascending=False)
engagement.head(3), engagement.tail(3)

(    gender  age       mean  count
 5   FEMALE   16  49.259259     54
 2   FEMALE   13  48.222222     90
 14    MALE   17  47.777778     18,
    gender  age       mean  count
 11   MALE   14  37.567568     74
 15   MALE   18  32.307692     13
 16   MALE   19  23.333333      6)

In [147]:
# female peak
fpeak = engagement[engagement['gender']=='FEMALE'].iloc[0, :]
xf, yf = fpeak['age'], fpeak['mean']
# male peak
mpeak = engagement[engagement['gender']=='MALE'].iloc[0, :]
xm, ym = mpeak['age'], mpeak['mean']

In [202]:
hist_att_age_gen = px.bar(engagement.sort_values('age'), 
             x='age', y='mean', color='gender',
             barmode='group', text='mean', # shows the mean value on each bar
             color_discrete_map={'MALE': '#87ceeb', 'FEMALE': '#ffb6c1'},
             title="Average Attendance Rate by Age and Gender")  
hist_att_age_gen.update_traces(texttemplate='%{text:.1f}%',  # 1 decimal place
                  textposition='outside')
hist_att_age_gen.update_xaxes(dtick=1)  # show every age on x-axis
hist_att_age_gen.update_layout(xaxis_title='Age', yaxis_title='Average Attendance Rate (%)',
                  legend_title='Gender', autosize=True, )
                  # margin=dict(l=20, r=20, t=40, b=20))

## add annotation for clearer insights
# female peak
hist_att_age_gen.add_annotation(x=xf+0.2, y=yf+2.5, text="Female\npeak", 
                   showarrow=True, arrowhead=2, ax=0, ay=-20, arrowcolor="red",
                   font=dict(size=12, color="red"))
hist_att_age_gen.add_shape(type="rect", x0=xf, x1=xf+0.4, y0=yf+0.2, y1=yf+3,
              fillcolor="red", opacity=0.2, line_width=0)
# male peak
hist_att_age_gen.add_annotation(x=xm-0.2, y=ym+2.5, text="Male\npeak", 
                   showarrow=True, arrowhead=2, ax=0, ay=-20, arrowcolor="blue",
                   font=dict(size=12, color="blue"))
hist_att_age_gen.add_shape(type="rect", x0=xm, x1=xm-0.4, y0=ym+0.2, y1=ym+3,
              fillcolor="blue", opacity=0.2, line_width=0)
# # arrow: male attendance drop after peak
# hist_att_age_gen.add_annotation(x=19.2, y=32, ax=17.2, ay=60, xref='x', yref='y', axref='x', ayref='y', text="",
#                    showarrow=True, arrowhead=2, arrowcolor="blue", arrowwidth=1.5)
# # arrow: female attendance drop after peak
# hist_att_age_gen.add_annotation(x=17.8, y=54, ax=15.8, ay=61.5, xref='x', yref='y', axref='x', ayref='y', text="",
#                    showarrow=True, arrowhead=2, arrowcolor="red", arrowwidth=1.5)

hist_att_age_gen.show()

In [90]:
var = att_age_gender.agg(['mean', 'std', 'var']).reset_index()
var = var.sort_values(by=['mean'], ascending=False)
# Higher std = more irregular attendance within that group.

## Arrival rush by min

In [91]:
df.head(3)

,event_id,event_name,event_time,participant_id,participant_name,checkin_time,participant_birth,participant_gender,participant_age
0,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,59fa3bcd-4903-4349-ac44-4af3352efc93,Felipe,2026-03-12 18:56:04.693167-03:00,2009-11-19,MALE,16
1,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,346c4a15-bf18-4917-9877-48df46d84fdc,Felipe,2026-03-12 19:13:15.777129-03:00,2012-07-19,MALE,13
2,171efb44-3973-4f55-bad0-8d32f53128ea,12/03,2026-03-12 18:30:00,13762581-ab39-4b01-ab0b-274ac8c6bceb,Antônio,2026-03-12 19:13:40.541487-03:00,2008-09-13,MALE,17


In [118]:
checkin_info = df[['event_name', 'checkin_time']].copy()
checkin_info['hour'] = checkin_info['checkin_time'].dt.hour
checkin_info['minute'] = checkin_info['checkin_time'].dt.minute
checkin_info['hr_min'] = checkin_info['checkin_time'].dt.strftime('%H:%M')
checkin_info['hr_min_3'] = checkin_info['checkin_time'].dt.floor('3min').dt.strftime('%H:%M')
checkin_info['hr_min_5'] = checkin_info['checkin_time'].dt.floor('5min').dt.strftime('%H:%M')
checkin_info.head()

,event_name,checkin_time,hour,minute,hr_min,hr_min_3,hr_min_5
0,12/03,2026-03-12 18:56:04.693167-03:00,18,56,18:56,18:54,18:55
1,12/03,2026-03-12 19:13:15.777129-03:00,19,13,19:13,19:12,19:10
2,12/03,2026-03-12 19:13:40.541487-03:00,19,13,19:13,19:12,19:10
3,12/03,2026-03-12 19:17:05.458682-03:00,19,17,19:17,19:15,19:15
4,12/03,2026-03-12 19:17:35.558127-03:00,19,17,19:17,19:15,19:15


In [93]:
checkin_info["hr_min"].describe()

count       924
unique       54
top       19:30
freq         56
Name: hr_min, dtype: object

In [168]:
# Check-in minute rush by event
min_rush_event = checkin_info.groupby(['event_name', 'hr_min'], as_index=False).size()
# min_rush = min_rush.sort_values(by=['hr_min'])
time_order = sorted(min_rush_event['hr_min'].unique(), key=lambda x: pd.to_datetime(x, format='%H:%M'))
min_fig = px.line(min_rush_event, x='hr_min', y='size', 
                  category_orders={'hr_min': time_order},  # ensure x-axis is in chronological order
                  color='event_name', markers=True,
                  title='Check-in Count by Time per Event')
min_fig.update_xaxes(dtick=3, tickangle=30)  
min_fig.update_layout(xaxis_title='Time (Hour:Minute)', yaxis_title='Number of Check-ins')

# add vertical lines for key time points 
time_ticks = ['19:15', '19:25', '19:45', '19:50']
for t in time_ticks:
    min_fig.add_vline(x=t, line_dash='dash', line_color='grey', line_width=1.5)
min_fig.add_trace(go.Scatter(x=time_ticks, y=[None] * len(time_ticks), xaxis='x2', 
                             showlegend=False, hoverinfo='skip', mode='markers', marker=dict(opacity=0)))
min_fig.update_layout(xaxis2=dict(overlaying='x', side='top', matches='x', 
                                  tickvals=[time_order.index(t) for t in time_ticks],  # numeric positions of those 4 times
                                  ticktext=time_ticks, showgrid=False,))
for p in [10, 20, 30]:
    min_fig.add_hline(y=p, line_dash='dash', line_color='grey', line_width=1.5)
min_fig.add_vline(x='19:30', line_dash='solid', line_color='black', line_width=2)
min_fig

In [171]:
# Check-in minute rush & estimate number of staff needed per time slot
# METHOD 1: by taking the max check-in count across events for each time slot (to make sure there's enough staff for the busiest minute in the time slot)
min_rush_contour = min_rush_event.groupby('hr_min', as_index=False)['size'].max()
contour_fig = px.bar(min_rush_contour, x='hr_min', y='size', 
                      category_orders={'hr_min': time_order},
                      title='Max Check-in Count Across Events by Time')
contour_fig.update_xaxes(dtick=3, tickangle=30)
contour_fig.update_layout(xaxis_title='Time (Hour:Minute)', yaxis_title='Number of Check-ins')
# dashed vertical lines
time_labels = time_order
time_ticks = ['19:20', '19:30', '19:40', '19:50']
for t in time_ticks:
    idx = time_labels.index(t)
    contour_fig.add_vline(x=idx-0.5, line_dash='dash', line_color='grey', line_width=1.5)
contour_fig.add_trace(go.Scatter(x=time_ticks, y=[None] * len(time_ticks), xaxis='x2', 
                             showlegend=False, hoverinfo='skip', mode='markers', marker=dict(opacity=0)))
contour_fig.update_layout(xaxis2=dict(overlaying='x', side='top', matches='x', 
                                  tickvals=[time_order.index(t) for t in time_ticks],  # numeric positions of those 4 times
                                  ticktext=time_ticks, showgrid=False,))
# event start time vertical line
def get_x_between(t1, t2):
    return (time_labels.index(t1) + time_labels.index(t2)) / 2
contour_fig.add_vline(x=get_x_between('19:29', '19:30'), line_dash='solid', line_color='red', line_width=2)
contour_fig.add_annotation(x=get_x_between('19:29', '19:30'), y=min_rush_contour['size'].max(), text="Event Starts")
contour_fig

1 person scans 10 QR code per minute:
* before 19:20 -> 1 person
* 19:20 - 19:30 -> 2 people
* 19:30 - 19:40 -> 3 people
* 19:40 - 19:50 -> 2 people
* after 19:50 -> 1 person


In [135]:
# Overall check-in time distribution
# METHOD 2: by average check-ins per event in each time slot
min_rush_all = checkin_info.groupby(['hr_min'], as_index=False).size()
min_rush_all['size'] = (min_rush_all['size']+num_events-1) // num_events  # average check-ins per event in each time slot

min_fig_all = px.bar(min_rush_all, x='hr_min', y='size', 
                  category_orders={'hr_min': time_order},  # ensure x-axis is in chronological order
                  title='Check-in Count by Time')
min_fig_all.update_xaxes(dtick=3, tickangle=30)  
min_fig_all.update_layout(xaxis_title='Time (Hour:Minute)', yaxis_title='Number of Check-ins')

# dashed vertical lines
for time in ['19:20', '19:30', '19:40', '19:50']:
    min_fig_all.add_vline(x=time, line_dash='dash', line_color='grey', line_width=1.5)

# solid vertical line
min_fig_all.add_vline(x='19:30', line_dash='solid', line_color='red', line_width=2)
min_fig_all

## Drop-out Risk Analysis

In [123]:
events = df[['event_id', 'event_name', 'event_time']].drop_duplicates().reset_index(drop=True)
events = events.sort_values(by='event_time').reset_index(drop=True)

dropout_threshold = 4 # if a person hasn't attended the most recent {dropout_threshold} events, consider them at risk of dropping out
recent_event_ids = events.tail(dropout_threshold)["event_id"].tolist()
attended = df[df['event_id'].isin(recent_event_ids)]['participant_id'].unique()
dropout_risk = participants[~participants['id'].isin(attended)]
print(f"Number of participants at risk of dropping out: {len(dropout_risk)}")
print(f"Percentage: {len(dropout_risk)/len(participants):.2%}")

Number of participants at risk of dropping out: 97
Percentage: 14.26%


## Event Analysis

# Dashboard

In [179]:
import dash
from dash import dcc, html
app = dash.Dash(__name__)

In [ ]:
## === KPI cards ===

# (1) total participants & age range
kpi_1 = html.Div([
    html.P("Total Participants", style={"fontSize": "13px", "color": "gray", "margin": "0 0 6px"}),
    html.P(f"{num_participants}", style={"fontSize": "28px", "fontWeight": "500", "margin": "0"}),
    html.Hr(style={"border": "none", "borderTop": "0.5px solid #ccc", "margin": "10px 0 8px"}),
    html.P(f"Age range: {age_info['min']} - {age_info['max']}", style={"fontSize": "12px", "color": "gray", "margin": "0"}),
    ], style={"background": "#f5f5f5", "borderRadius": "8px", "padding": "1rem", "width": "200px"})
# (2) average age & median age
kpi_2 = html.Div([
    html.P("Average Age", style={"fontSize": "13px", "color": "gray", "margin": "0 0 6px"}),
    html.P(f"{age_info['mean']:.1f}", style={"fontSize": "28px", "fontWeight": "500", "margin": "0"}),
    html.Hr(style={"border": "none", "borderTop": "0.5px solid #ccc", "margin": "10px 0 8px"}),
    html.P(f"Median Age: {age_info['median']}", style={"fontSize": "12px", "color": "gray", "margin": "0"}),
    ], style={"background": "#f5f5f5", "borderRadius": "8px", "padding": "1rem", "width": "200px"})
# (3) gender distribution
kpi_3 = html.Div([
    html.P("Gender Distribution", style={"fontSize": "13px", "color": "gray", "margin": "0 0 6px"}),
    html.Div([ # percentages
        html.Span(f"{gender_pct['female']:.1f}% F", style={"fontSize": "20px", "fontWeight": "500"}),
        html.Span(f"{gender_pct['male']:.1f}% M", style={"fontSize": "20px", "fontWeight": "500"}),
        ], style={"display": "flex", "justifyContent": "space-between", "margin": "4px 0 8px"}),
    html.Div([ # split bar
        html.Div(style={"width": f"{gender_pct['female']}%", "height": "6px", "background": "#AFA9EC"}),
        html.Div(style={"width": f"{gender_pct['male']}%", "height": "6px", "background": "#5DCAA5"}),
        ], style={"display": "flex", "borderRadius": "4px", "overflow": "hidden"}),
    ], style={"background": "#f5f5f5", "borderRadius": "8px", "padding": "1rem", "width": "200px"})

## === Layout ===
app.layout = html.Div([
    # Title
    html.H1(f"Parusya Dashboard: {GNAME}", 
            style={"textAlign": "center", "fontSize": "48px", "fontWeight": "500", "margin": "2rem 0 1rem",}),
    # KPI row
    html.Div([ kpi_1, kpi_2, kpi_3 ], 
             style={"display": "flex", "justifyContent": "center", "gap": "16px"}),
    # # Charts row
    # charts row — 2 side by side
    html.Div([
        dcc.Graph(figure=bar_age, style={"flex": "1", "minWidth": "0"}),
        dcc.Graph(figure=hist_att_age_gen, style={"flex": "1", "minWidth": "0"}),
    ], style={"display": "flex", "gap": "6px", "maxWidth": "100%", "margin": "1rem 6rem 1rem"}),
    # Full-width chart
    dcc.Graph(figure=contour_fig, style={"maxWidth": "80%", "margin": "0 auto"}),
])

In [ ]:
if __name__ == "__main__":
    """ Instruction: 
    open http://127.0.0.1:8050/ in the browser to view the dashboard
    """
    app.run(debug=True)
    

report: https://excited-fernleaf-3ff.notion.site/DA-Report-for-Parusya-33b278eaa1d980b1961fd2cccf6d0f4e?source=copy_link